# 02 · Cleaning and exploration

Loads the four exports (course history, attendance, and two state-test files), cleans them, and builds one table with a row per student. Set `DATA_SOURCE` to `"real"` for local work on the district data, or `"synthetic"` for the published version. Outputs in this repository always come from the synthetic data.

In [1]:
from pathlib import Path
import pandas as pd

# "synthetic" reproduces the published results. "real" requires the private district data, which is not included.
DATA_SOURCE = "synthetic"
DATA_DIR = Path("../data") if DATA_SOURCE == "real" else Path("../synthetic_data")

## 1. Load the exports

In [2]:
grades = pd.read_excel(DATA_DIR / "grades.xlsx")
attendance = pd.read_csv(DATA_DIR / "attendance.csv")
tests_a = pd.read_excel(DATA_DIR / "tests_a.xlsx")
tests_b = pd.read_excel(DATA_DIR / "tests_b.xlsx")

## 2. Keep only the columns the analysis needs

This is a privacy step as much as a cleaning step. The raw exports include names, state IDs, addresses, and program flags that the analysis does not need. I use a list of columns to keep rather than a list to drop, so a sensitive column cannot slip through just because I forgot to name it.

`Grade` is also left out of attendance and tests. In those exports it is the student's grade at the time of the extract (always 12), not the grade in the year of the record. Real grade levels come from the course history.

In [3]:
GRADES_KEEP = ["Student ID", "Calendar Month", "Calendar Year", "Course ID", "Grade",
               "Course Title", "Credit Attempted", "Credit Completed", "School Year", "Mark"]
ATTENDANCE_KEEP = ["Student ID", "Days Absent", "Days Membership", "School Year",
                   "OrganizationName"]
TESTS_KEEP = ["Student ID", "Admin Date", "Test Name", "Part Number", "Part Description",
              "Scale Score", "Performance Level for Grade", "Performance Level for Benchmark"]

grades = grades[GRADES_KEEP]
attendance = attendance[ATTENDANCE_KEEP]
tests_a = tests_a[TESTS_KEEP]
tests_b = tests_b[TESTS_KEEP]

## 3. Remove duplicate rows

All the exports contain repeated rows. Two patterns explained most of them:

- **Course history** repeated some records with the student's name spelled differently, so the rows were not identical until names were dropped.
- **State tests** were repeated once per enrollment record, each copy carrying a different enrollment's Enter and Leave dates. Dropping those columns (step 2) turned the copies into exact duplicates.

That is why the column step comes first: deduplicating on the full raw exports would have kept thousands of copies. The two test files also overlap for some students, so they are combined before deduplicating.

In [4]:
tests = pd.concat([tests_a, tests_b], ignore_index=True)

def dedupe(df, name):
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"{name}: {before:,} -> {after:,}")
    return df

grades = dedupe(grades, "grades")
attendance = dedupe(attendance, "attendance")
tests = dedupe(tests, "tests")

grades: 73,688 -> 54,100
attendance: 45,730 -> 5,876
tests: 37,530 -> 26,398


Attendance shrank by almost 90% and course history by about a quarter, so most of the raw attendance export was copies.

## 4. First look at the course history

In [5]:
grades.shape

(54100, 10)

In [6]:
grades.head()

,Student ID,Calendar Month,Calendar Year,Course ID,Grade,Course Title,Credit Attempted,Credit Completed,School Year,Mark
0,1180315,2-February,2023-2023,C1050,12,Career Prep,0.25,0.00,2022-2022,F
1,1406756,11-November,2010-2010,C1025,09,Algebra 1,0.25,0.25,2010-2010,C
2,1724587,6-June,2011-2011,C1033,10,LA 10,0.50,0.50,2010-2010,A
3,1062847,1-January,2015-2015,C1006,06,Social Studies 6,0.50,0.50,2014-2014,D
4,1735463,1-January,2013-2013,C1010,07,Focus 7,0.50,0.00,2012-2012,F


In [7]:
grades.dtypes

Student ID            int64
Calendar Month          str
Calendar Year           str
Course ID               str
Grade                   str
Course Title            str
Credit Attempted    float64
Credit Completed    float64
School Year             str
Mark                    str
dtype: object

Every column except the IDs and credits is stored as text, including the year columns, so they will need converting before any date math. The year columns also have an unusual format, so I look at them next.

## 5. What the school-year labels mean

In [8]:
grades["School Year"].value_counts().sort_index()

School Year
2007-2007     796
2008-2008    1326
2009-2009    1870
2010-2010    2695
2011-2011    3677
2012-2012    4555
2013-2013    4351
2014-2014    4325
2015-2015    4238
2016-2016    4434
2017-2017    4536
2018-2018    4035
2019-2019    3909
2020-2020    3408
2021-2021    2920
2022-2022    1958
2023-2023    1067
Name: count, dtype: int64

Every `School Year` label repeats a single year ("2015-2015"), so the label does not say whether it means the fall or the spring of that school year.

In [9]:
grades["Calendar Month"].value_counts()

Calendar Month
6-June         20099
1-January      12562
4-April         7155
11-November     7143
2-February      7141
Name: count, dtype: int64

Grades post at the end of each grading period. The main high school runs semesters (January, June), and the alternative program appears to run quarters (November, February, April, June). So grades post in both halves of the school year, which gives a way to test the label: if I take one label and look at when its grades posted, fall and spring grades should land in different calendar years.

In [10]:
grades.loc[grades["School Year"] == "2015-2015", ["Calendar Month", "Calendar Year"]].value_counts()

Calendar Month  Calendar Year
6-June          2016-2016        1633
1-January       2016-2016        1088
11-November     2015-2015         507
4-April         2016-2016         506
2-February      2016-2016         504
Name: count, dtype: int64

For the "2015-2015" label, November grades posted in 2015 and January to June grades posted in 2016. So the label is the **fall** year: "2015-2015" means the 2015-16 school year. Next I check that this holds for every label, not just one.

Each course row has a `School Year` label and a `Calendar Year`, the year the grade posted. If the label is the fall year, spring grades (January to June) should post one calendar year after the label, and fall grades (August to December) in the same year.

In [11]:
check = grades.assign(
    label_year=grades["School Year"].str[:4].astype(float),
    calendar_year=grades["Calendar Year"].str[:4].astype(float),
    month=grades["Calendar Month"].str.split("-").str[0].astype(float),
)
check["gap"] = check["calendar_year"] - check["label_year"]
pd.crosstab(check["month"], check["gap"])

gap,0.0,1.0
month,,
1.0,0,12561
2.0,0,7140
4.0,0,7153
6.0,0,20097
11.0,7143,0


Every spring grade posts one year after its label and every fall grade posts in the same year as its label, with no exceptions. So the label is the fall year: `sy` = label + 1.

The exports label school years three different ways. Next I convert all of them to `sy`, the spring year in which a school year ends (2016 means 2015-16).

In [12]:
# grades: "2015-2015" is the FALL year, so the school year ends one year later
grades["sy"] = pd.to_numeric(grades["School Year"].str[:4]) + 1

# attendance: "2015-2016" and "2016" both end in the spring year, the last four characters
attendance["sy"] = pd.to_numeric(attendance["School Year"].astype(str).str[-4:])

print("grades sy:", grades["sy"].min(), "to", grades["sy"].max())
print("attendance sy:", attendance["sy"].min(), "to", attendance["sy"].max())

grades sy: 2008 to 2024
attendance sy: 2007 to 2024


For attendance, I treat a single-year label ("2016") as the spring year, the same as the second year in a range ("2015-2016"). The attendance export has no term dates to test this against, so it is an assumption. A one-year error here would shift absence rates to the wrong grade level, so it is worth confirming with the district if this model were used.

In [13]:
grades[["School Year", "sy"]].drop_duplicates().head()

,School Year,sy
0,2022-2022,2023
1,2010-2010,2011
3,2014-2014,2015
4,2012-2012,2013
5,2011-2011,2012


In [14]:
attendance[["School Year", "sy"]].drop_duplicates().head(10)

,School Year,sy
0,2015-2016,2016
1,2012-2013,2013
2,2011-2012,2012
3,2019-2020,2020
4,2018,2018
7,2008-2009,2009
8,2020-2021,2021
10,2022-2023,2023
11,2015,2015
13,2009-2010,2010


## 6. Assign each student a senior cohort

Every student in the data reached 12th grade. A student's cohort is the school year (`sy`) of their senior year. If a student has two senior years, I use the later one. The `Grade` column in the course history records the grade level for each course, so it can tell me when that was.

In [15]:
grades["Grade"].value_counts(dropna=False)

Grade
12     13691
11     10729
10      8696
09      7869
08      4755
07      4468
06      3838
08H       34
NaN       20
Name: count, dtype: int64

Grade levels run from 06 to 12, stored as text with a leading zero. There are also a few "08H" rows (8th graders earning high school credit) and a few blanks.

In [16]:
cohort = grades[grades["Grade"] == "12"].groupby("Student ID")["sy"].max()
cohort.head()

Student ID
1001754    2024
1002162    2020
1002616    2024
1004242    2023
1005060    2017
Name: sy, dtype: int64

In [17]:
print("students with a cohort:", len(cohort))
print("students in the course history:", grades["Student ID"].nunique())
cohort.value_counts().sort_index()

students with a cohort: 450
students in the course history: 450


sy
2013    44
2014    35
2015    43
2016    32
2017    31
2018    42
2019    30
2020    36
2021    36
2022    47
2023    36
2024    38
Name: count, dtype: int64

## 7. Outcome: earned 24 high school credits

A diploma requires 24 credits, including modified diplomas. The outcome is whether a student's high school credits (grades 9 to 12) add up to at least 24. 8th-grade high school credits ("08H") are not counted.

In [18]:
hs_credits = grades[grades["Grade"].isin(["09", "10", "11", "12"])].groupby("Student ID")["Credit Completed"].sum()

hs_credits.describe()

count    450.000000
mean      22.109444
std        5.238211
min        5.000000
25%       19.500000
50%       23.250000
75%       25.687500
max       39.750000
Name: Credit Completed, dtype: float64

In [19]:
earned_24 = (hs_credits >= 24).astype(int)
earned_24.value_counts(normalize=True)

Credit Completed
0    0.566667
1    0.433333
Name: proportion, dtype: float64

About 43% of students earned 24 credits, so the two outcomes are close to balanced.

In [20]:
students = pd.DataFrame({
    "cohort": cohort,
    "hs_credits": hs_credits,
    "earned_24": earned_24,
})
students.head()

,cohort,hs_credits,earned_24
Student ID,,,
1001754,2024,24.25,1
1002162,2020,23.25,0
1002616,2024,21.00,0
1004242,2023,16.00,0
1005060,2017,25.00,1


In [21]:
students.groupby("cohort")["earned_24"].agg(["count", "mean"])

,count,mean
cohort,,
2013,44,0.340909
2014,35,0.314286
2015,43,0.418605
2016,32,0.562500
2017,31,0.483871
2018,42,0.404762
2019,30,0.500000
2020,36,0.416667
2021,36,0.416667


The rate ranges from about 30% to 55% across cohorts with no steady trend up or down. With 30 to 47 students per cohort, swings of that size are expected from chance alone.

## 8. Course marks

Before building 9th-grade features, I look at what marks appear and decide how each one counts.

In [22]:
grades["Mark"].value_counts(dropna=False)

Mark
F     10015
C      9270
A      8307
D      8097
P      7328
B      7229
NG     1939
I       950
NP      637
W       172
WF      156
Name: count, dtype: int64

In [23]:
GPA_POINTS = {"A": 4, "B": 3, "C": 2, "D": 1, "F": 0}
PASSING = ["A", "B", "C", "D", "P"]

grades["points"] = grades["Mark"].map(GPA_POINTS)
grades["passed"] = grades["Mark"].isin(PASSING).astype(int)

grades[["Mark", "points", "passed"]].drop_duplicates().sort_values("Mark")

,Mark,points,passed
2,A,4.0,1
10,B,3.0,1
1,C,2.0,1
3,D,1.0,1
0,F,0.0,0
136,I,NaN,0
23,NG,NaN,0
115,NP,NaN,0
8,P,NaN,1
116,W,NaN,0


Passing marks are A through D and P. Everything else (F, NP, NG, I, W, WF) counts as not passed. GPA uses letter grades only, so pass/fail courses do not affect it. The original analysis scored P as 0, the same as an F, which penalized students in the alternative program, where pass/fail grading is common.

## 9. Features: 9th-grade course performance

The model predicts from what a school knows at the end of a student's first 9th-grade year. Students who repeated 9th grade have two 9th-grade years, so I keep only the first.

9th-grade credits are part of the 24-credit outcome, but they are known at the end of 9th grade, which is the point of prediction. The model uses only what a school would actually know then.

In [24]:
ninth = grades[grades["Grade"] == "09"]
first_9th = ninth.groupby("Student ID")["sy"].min().rename("first_9th_sy")
ninth = ninth.merge(first_9th, left_on="Student ID", right_index=True)
ninth = ninth[ninth["sy"] == ninth["first_9th_sy"]]

print("students with 9th-grade records:", ninth["Student ID"].nunique())

students with 9th-grade records: 444


In [25]:
features_9 = ninth.groupby("Student ID").agg(
    gpa_9=("points", "mean"),
    pass_rate_9=("passed", "mean"),
    credits_9=("Credit Completed", "sum"),
)
features_9.describe()

,gpa_9,pass_rate_9,credits_9
count,444.000000,444.000000,444.000000
mean,1.936440,0.771992,5.385698
std,0.965749,0.223706,1.590400
min,0.000000,0.035714,0.250000
25%,1.214286,0.666667,4.500000
50%,1.928571,0.857143,6.000000
75%,2.714286,0.933333,6.500000
max,4.000000,1.000000,7.000000


In their first 9th-grade year, the typical student had a GPA just under 2.0, passed 86% of courses, and earned 6 credits.

## 10. Attendance

Attendance should have one row per student per school year.

In [26]:
attendance.shape

(5876, 6)

In [27]:
attendance.head()

,Student ID,Days Absent,Days Membership,School Year,OrganizationName,sy
0,1741730,5.0,165,2015-2016,Northgate Alternative Program,2016
1,1094578,5.0,174,2012-2013,Riverside Elementary,2013
2,1106785,8.0,171,2011-2012,Riverside Elementary,2012
3,1678945,9.0,174,2019-2020,Northgate Alternative Program,2020
4,1077292,11.0,152,2018,Northgate High School,2018


In [28]:
attendance.dtypes

Student ID            int64
Days Absent         float64
Days Membership       int64
School Year             str
OrganizationName        str
sy                    int64
dtype: object

In [29]:
attendance[["Days Absent", "Days Membership"]].describe()

,Days Absent,Days Membership
count,5876.000000,5876.000000
mean,14.363513,160.693329
std,12.987153,33.347316
min,0.000000,0.000000
25%,5.000000,165.000000
50%,11.000000,169.000000
75%,19.000000,172.000000
max,117.000000,175.000000


Each row is a student's attendance for one school year at one school: days absent and days enrolled ("membership"). Most membership values are close to a full year of about 170 days, but some are much lower (students who enrolled mid-year) and some are 0.

In [30]:
attendance["School Year"].value_counts().sort_index()

School Year
2006-2007    219
2007-2008    255
2008          77
2008-2009    275
2009          83
2009-2010    325
2010          83
2010-2011    364
2011         112
2011-2012    403
2012         117
2012-2013    415
2013         117
2013-2014    368
2014         114
2014-2015    334
2015         118
2015-2016    309
2016          95
2016-2017    281
2017          97
2017-2018    254
2018          90
2018-2019    215
2019          73
2019-2020    180
2020          64
2020-2021    154
2021          52
2021-2022    121
2022-2023     74
2023-2024     38
Name: count, dtype: int64

School years appear in two formats: a range ("2015-2016") and a single year ("2016"). Both convert to the same `sy`, but having two formats in one export is a sign the data came from more than one source, so I check whether any student-year appears twice.

In [31]:
attendance["OrganizationName"].value_counts()

OrganizationName
Harbor Middle School             1352
Riverside Elementary             1214
Northgate High School            1072
Northgate Alternative Program    1017
Other In-State Schools            856
Hillcrest Elementary              340
Desert Valley High School           9
Prairie View High School            7
Pine Hollow High School             5
Canyon Ridge High School            4
Name: count, dtype: int64

Records cover every school a student attended in-state, from elementary school on, not just the high school, which is why most students have ten or more years of attendance.

In [32]:
attendance["single_label"] = ~attendance["School Year"].astype(str).str.contains("-")
print(attendance["single_label"].value_counts())

rows_per_year = attendance.groupby(["Student ID", "sy"]).size()
rows_per_year.value_counts()

single_label
False    4584
True     1292
Name: count, dtype: int64


1    3578
2    1149
Name: count, dtype: int64

In [33]:
doubled = rows_per_year[rows_per_year > 1].index
pairs = attendance.set_index(["Student ID", "sy"]).loc[doubled]
pairs.groupby(level=[0, 1])["Days Membership"].sum().describe()

count    1149.000000
mean      322.922541
std        36.499130
min       151.000000
25%       324.000000
50%       331.000000
75%       338.000000
max       350.000000
Name: Days Membership, dtype: float64

Some student-years have two rows, almost always one with each label format, at different schools, and each with close to a full year of membership. Together they imply more than 300 days enrolled in one year, which is impossible, and students can only be enrolled at one high school at a time. I keep the range-format row, the export's standard format, and drop the single-year row. If both rows share a format, I keep the one with more membership days.

In [34]:
attendance = (attendance
              .sort_values(["single_label", "Days Membership"], ascending=[True, False])
              .drop_duplicates(subset=["Student ID", "sy"], keep="first"))

print("rows:", len(attendance))
print("student-years with 2+ rows:", (attendance.groupby(["Student ID", "sy"]).size() > 1).sum())

rows: 4727
student-years with 2+ rows: 0


## 11. Feature: 9th-grade absence rate

Rows with 0 days enrolled are dropped first, since no absence rate can be computed for them.

In [35]:
# drop rows where the student was enrolled 0 days (no rate can be computed)
attendance = attendance[attendance["Days Membership"] > 0].copy()

# share of enrolled days missed
attendance["absence_rate"] = attendance["Days Absent"] / attendance["Days Membership"]
attendance["absence_rate"].describe()

count    4526.000000
mean        0.089183
std         0.076400
min         0.000000
25%         0.035714
50%         0.066667
75%         0.118644
max         0.672619
Name: absence_rate, dtype: float64

In [36]:
att_9 = attendance.merge(first_9th, left_on="Student ID", right_index=True)
att_9 = att_9[att_9["sy"] == att_9["first_9th_sy"]]

absence_9 = att_9.set_index("Student ID")["absence_rate"].rename("absence_rate_9")

print("students with 9th-grade attendance:", len(absence_9))
absence_9.describe()

students with 9th-grade attendance: 431


count    431.000000
mean       0.104651
std        0.081561
min        0.000000
25%        0.045977
50%        0.087719
75%        0.138554
max        0.559524
Name: absence_rate_9, dtype: float64

## 12. Join features onto the student table

In [37]:
students = students.join(features_9).join(absence_9)
students.head()

,cohort,hs_credits,earned_24,gpa_9,pass_rate_9,credits_9,absence_rate_9
Student ID,,,,,,,
1001754,2024,24.25,1,2.928571,0.928571,6.5,0.022857
1002162,2020,23.25,0,2.538462,0.800000,5.5,0.058480
1002616,2024,21.00,0,1.642857,0.857143,6.0,0.017647
1004242,2023,16.00,0,1.461538,0.928571,6.5,0.183432
1005060,2017,25.00,1,1.500000,0.866667,6.0,0.132948


In [38]:
students.isna().sum()

cohort             0
hs_credits         0
earned_24          0
gpa_9              6
pass_rate_9        6
credits_9          6
absence_rate_9    19
dtype: int64

A few students have no 9th-grade course or attendance records, mostly transfer students whose earlier records are not in the district's export. How to handle them is a modeling decision for notebook 03.

## 13. Feature: 8th-grade state test results

Tests run from grade 3 through high school in reading/ELA, math, and science. The state switched from a legacy test to Smarter Balanced (SBA) in 2015, and the two use different score scales, so scale scores cannot be compared across them. Both report whether a student met the standard, so I use that. The file also includes graduation-requirement records (resume, education plan, career experiences) that are not tests. They are excluded, since they are completed close to the outcome.

In [39]:
tests.shape

(26398, 8)

In [40]:
tests.head()

,Student ID,Admin Date,Test Name,Part Number,Part Description,Scale Score,Performance Level for Grade,Performance Level for Benchmark
0,1221837,2009-05-28,"LEG Math, Grade 05",4,Geometry,253,E,E
1,1742073,2011-05-16,"LEG Math, Grade 07",3,Measurement,225,M,M
2,1027194,2010-05-24,"LEG Science, Grade 05",3,Physical Science,221,M,M
3,1323545,2014-04-16,"LEG Reading, Grade 08",3,Demonstrate General Understanding,215,D,D
4,1382493,2008-04-21,"LEG Reading, Grade 04",1,Total,220,M,M


In [41]:
tests["Test Name"].value_counts()

Test Name
LEG Math, Grade 03            1615
LEG Math, Grade 04            1425
LEG Science, Grade 05         1348
SBA ELA HS                    1325
LEG Math, Grade 05            1255
LEG Science, Grade 08         1236
LEG Reading, Grade 03         1200
LEG Reading, Grade 04         1148
LEG Math, Grade 06            1140
SBA MA HS                     1060
LEG Math, Grade 07            1020
LEG Reading, Grade 05          996
LEG Math, Grade 08             965
LEG Reading, Grade 06          936
SBA ELA Gr 8                   905
LEG Science, Grade 11          848
LEG Reading, Grade 07          836
LEG Reading, Grade 08          788
SBA ELA Gr 7                   750
SBA MA Gr 8                    684
SBA MA Gr 7                    624
SBA ELA Gr 6                   585
LEG Math, Grade 09-12          520
SBA ELA Gr 5                   485
SBA MA Gr 6                    472
LEG Reading, Grade 09-12       416
SBA MA Gr 5                    380
SBA ELA Gr 4                   290
SBA MA Gr 

In [42]:
totals = tests[tests["Part Description"] == "Total"].copy()
name = totals["Test Name"]
totals["subject"] = None
totals.loc[name.str.contains("Reading|ELA"), "subject"] = "ela"
totals.loc[name.str.contains("Math|MA "), "subject"] = "math"

grade8 = totals[name.str.contains("Grade 08|Gr 8") & totals["subject"].notna()].copy()
grade8["Test Name"].value_counts()

Test Name
LEG Reading, Grade 08    197
LEG Math, Grade 08       193
SBA ELA Gr 8             181
SBA MA Gr 8              171
Name: count, dtype: int64

In [43]:
grade8["Performance Level for Benchmark"].value_counts()

Performance Level for Benchmark
D    173
M    154
2    104
1     92
3     87
4     69
E     63
Name: count, dtype: int64

The legacy test reports D, M, and E (Does not meet, Meets, Exceeds). Smarter Balanced reports levels 1 to 4, where 3 and 4 meet the standard. I count M, E, 3, and 4 as meeting it, which puts both test eras on the same yes/no scale.

In [44]:
MEETS = ["M", "3", "4", "E"]
grade8["met"] = grade8["Performance Level for Benchmark"].astype(str).isin(MEETS).astype(int)

tests_8 = (grade8.pivot_table(index="Student ID", columns="subject", values="met", aggfunc="max")
           .add_prefix("met_").add_suffix("_8"))
tests_8.describe()

subject,met_ela_8,met_math_8
count,378.000000,364.000000
mean,0.497354,0.508242
std,0.500656,0.500620
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,1.000000
75%,1.000000,1.000000
max,1.000000,1.000000


About half of students met the standard in each subject.

In [45]:
students = students.join(tests_8)
students.isna().sum()

cohort             0
hs_credits         0
earned_24          0
gpa_9              6
pass_rate_9        6
credits_9          6
absence_rate_9    19
met_ela_8         72
met_math_8        86
dtype: int64

About 70 to 90 students have no 8th-grade result in a subject: students who arrived after 8th grade or were not tested that year. Notebook 03 has to handle these, along with the few students missing 9th-grade records.

## 14. Save the student table

One row per student: cohort, the outcome, and features known by the end of 9th grade. Notebook 03 loads this file.

In [46]:
print(students.shape)
students.to_csv(DATA_DIR / "students.csv")

(450, 9)


## Summary

The raw exports were heavily duplicated, mostly because the test export repeated each result once per enrollment record and the course export repeated records under name variants. School-year labels used three formats, which I converted to a single scale after checking what each one meant. The outcome is whether a student earned the 24 credits a diploma requires. The six features (9th-grade GPA, pass rate, credits, and absence rate, plus 8th-grade ELA and math results) use only what a school would know at the end of a student's first 9th-grade year.